### Problem 0: GitHub Issues

# Problem 1: Representing SHA-256 Data in Python

The code for this problem is in `problem1.py`. This file explains the choices behind it.

## Background

SHA-256 is defined in terms of 32-bit words, 512-bit message blocks and big-endian byte order. Python's built-in `int` has arbitrary precision, so it never overflows and has no fixed size. That doesn't match the standard on its own, so I looked at how to represent each piece of data. Because Problems 2 and 3 require NumPy, I use NumPy's fixed-width `uint32` type for words from the start, so the whole project uses one consistent convention.

## Summary

| Concept | Representation |
|---|---|
| 32-bit word | `np.uint32` |
| Sequence of 32-bit words | NumPy array with `dtype=np.uint32` |
| Input message | `bytes` |
| 512-bit message block | 64 bytes, read as an array of 16 `uint32` words |
| Final 256-bit hash | 8 `uint32` words, then 32 `bytes`, then 64 hex characters |
| Big-endian integers | `int.from_bytes(..., "big")` and the `">u4"` dtype |

## 1. 32-bit words

**Choice:** `np.uint32`.

SHA-256 adds numbers modulo 2^32, and its shifts and rotations work on exactly 32 bits. A Python `int` would grow past 32 bits, so I would have to mask the result after every operation and it would be easy to forget one. `np.uint32` is always 32 bits and wraps around by itself, so the type enforces the rule from the standard.

In `problem1.py`, section 1 shows that `0xFFFFFFFF + 1` becomes `0x100000000` as a Python int but wraps to `0` as a `uint32`. It also defines a small `rotr` function (rotate right), which SHA-256 uses heavily. The inputs are wrapped in `np.uint32(...)` because mixing a `uint32` with a plain Python int can make NumPy switch to a larger integer type and give wrong rotations.

**Alternative considered:** a plain `int` with `& 0xFFFFFFFF`. It works and needs no library, but nothing stops a value from growing past 32 bits if the mask is forgotten.

## 2. Sequences of 32-bit words

**Choice:** a NumPy array with `dtype=np.uint32`.

SHA-256 uses several fixed-length sequences of words: the 16-word message block, the 64-word message schedule, the 8 hash values, the 8 working variables and the 64 round constants. They are all indexed by position, so an array fits well, and every element keeps the 32-bit wraparound behaviour.

The `dtype` must always be written explicitly. Without it NumPy picks a default (usually 64-bit) and the 32-bit guarantee is lost. Section 2 of the code builds the initial hash values and an empty 64-word schedule and checks their type and length.

## 3. Input messages

**Choice:** `bytes`.

The standard defines the message as a string of bits, but real data (files, network data, encoded text) arrives as whole bytes. `bytes` is immutable, indexable and sliceable. Text has to be encoded before hashing, and the encoding is part of the input: the letter "é" is two bytes in UTF-8 and one byte in Latin-1, so the hashes would differ. The message length in bits is `8 * len(msg)`, which is needed later for padding. Section 3 of the code shows this.

## 4. 512-bit message blocks

**Choice:** 64 bytes of the padded message, read as an array of 16 `uint32` words.

512 bits is 64 bytes, which is 16 words of 4 bytes each. Padding adds a single 1 bit (the byte `0x80`), then zero bytes until the length is 56 mod 64, then the original length in bits as a 64-bit big-endian number. The result is a multiple of 64 bytes, so it can be cut into blocks by slicing. Each block is converted to words with `np.frombuffer(block, dtype=">u4")`, which reads the bytes as big-endian, followed by `.astype(np.uint32)` to get ordinary native `uint32` values for calculation.

For the message "abc" this gives exactly one block. The first word is `0x61626380` (the bytes for a, b, c and the 1 bit) and the last word is 24 (the length in bits). A message of 56 bytes or more has no room for the length in its first block, so it needs a second block. Problem 4 handles that properly.

## 5. The final 256-bit hash value

**Choice:** 8 `uint32` words during the calculation, and at the end 32 `bytes` shown as 64 hex characters.

The algorithm finishes with the eight words H0 to H7. Writing each one big-endian and joining them gives the 256-bit digest. Bytes are the raw form, and hex is the form people read and compare. The byte order matters: converting with `astype(">u4")` before `tobytes()` guarantees big-endian output. Calling `tobytes()` directly would use the machine's native order, which is little-endian on most computers, and give the wrong digest. Section 5 of the code rebuilds the known SHA-256 of "abc" from its eight words and confirms it matches `hashlib`. `hashlib` is part of Python's standard library and is only used to check results.

## 6. Big-endian integers

**Choice:** `int.from_bytes(..., "big")` for single numbers, and the `">u4"` dtype for arrays of words.

SHA-256 puts the most significant byte first, so the bytes `61 62 63 80` make the number `0x61626380`. Reading them in little-endian order would give `0x80636261`, which is wrong for SHA-256. I always state the byte order explicitly so the result never depends on the machine the code runs on. Section 6 of the code compares both orders using Python ints and NumPy arrays.

## Design decisions

- **Words:** `np.uint32`, so wraparound at 32 bits happens automatically.
- **Word sequences:** arrays with `dtype=np.uint32`, so every element stays 32 bits.
- **Messages:** `bytes`, because that is what hashing works on.
- **Blocks:** 64-byte slices, turned into 16 words only when processed.
- **Hash:** 8 words, then 32 bytes, then 64 hex characters.
- **Byte order:** always explicit (`"big"` or `">u4"`), never the machine default.

## Running the code

Install the course requirements with `pip install -r requirements.txt`, then run `python problem1.py`. Only NumPy is needed.

### Problem 2: SHA-256 Bitwise Operations

### Problem 3: Generating the SHA-256 Constants

### Problem 4: Padding and Parsing Messages

### Problem 5: The SHA-256 Compression Function

### Problem 6: Complete SHA-256